# WW3_homework

至少选择3种机器学习方法，例如支撑向量机方法（SVM）、朴素贝叶斯、高斯分类等。

数据为：贵州茅台（600519）、宁德时代（300750）、药明康德（603259）、科大讯飞（002230）4只股票的2018年1月1日-2020年4月1日所有数据

## Step 1: Data Preparation

In [34]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

stock_files = {
    '贵州茅台':'Financial_Data/Guizhou Moutai.csv',
    '宁德时代':'Financial_data/Contemporary Amperex Technology.csv',
    '药明康德':'Financial_data/WuXi AppTec.csv',
    '科大讯飞':'Financial_data/iFLytek.csv'
}

all_stocks_data = {}
for stock_name, file_name in stock_files.items():
    try:
        df = pd.read_csv(file_name)
        df['stock_name'] = stock_name
        all_stocks_data[stock_name] = df
        print(f"{stock_name} Readin Complete.")
    except FileNotFoundError:
        print(f"{stock_name} NOT FOUND")

processed_data = {}
for stock_name, df in all_stocks_data.items():
    df = df.copy()

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')

    feature_columns = [
        'closing price',
        'highest price',
        'lowest price',
        'opening price',
        'previous close',
        'price change',
        'price change percentage',
        'turnover rate',
        'trading volume',
        'trading value',
        'total market capitalization',
        'float market capitalization',
        #'number of transactions' 贵州茅台和药明康德没有这些数据？？？
    ]

    df['target'] = (df['closing price'].shift(-1) > df['closing price']).astype(int)
    df = df.dropna(subset=['target'])
    #print(len(df))

    X = df[feature_columns].values
    y = df['target'].values
    #print(df.columns.tolist())
    #print(feature_columns)
    #print(X.shape, y.shape)

    split_idx = int(len(X)*0.7)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    processed_data[stock_name] = {
        'X': X,
        'y': y,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'scaler': scaler,
        'features': feature_columns,
        'dates': df['Date'].values
    }

    print(f"{stock_name}'s Data is READY")
    #print(X)
    

贵州茅台 Readin Complete.
宁德时代 Readin Complete.
药明康德 Readin Complete.
科大讯飞 Readin Complete.
贵州茅台's Data is READY
宁德时代's Data is READY
药明康德's Data is READY
科大讯飞's Data is READY


## Step 2: Run with SVM

In [35]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for stock_name, data in processed_data.items():
    X_train = data['X_train_scaled']
    X_test = data['X_test_scaled']
    y_train = data['y_train']
    y_test = data['y_test']

    kernels = ['linear', 'rbf', 'poly']
    c_values = [0.1, 1, 10]

    best_accuracy = 0
    best_params = {}

    for kernel in kernels:
        for C in c_values:
            try:
                svm = SVC(kernel=kernel, C=C, gamma='scale', class_weight='balance',
                           random_state=42)
                
                svm.fit(X_train, y_train)
                y_pred = svm.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)

                print(f"kernel:{kernel}, C={C}: {accuracy:.4f}")

                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_params = {'kernel':kernel, 'C':C}
            except:
                continue
    
    best_svm = SVC(**best_params, gamma='scale', random_state=42)
    best_svm.fit(X_train, y_train)
    y_pred = best_svm.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=["-1","+1"]))
    cm = confusion_matrix(y_test, y_pred)
    print(f"confusion matrix:{cm}")


              precision    recall  f1-score   support

          -1       0.51      0.96      0.66        84
          +1       0.25      0.01      0.02        80

    accuracy                           0.50       164
   macro avg       0.38      0.49      0.34       164
weighted avg       0.38      0.50      0.35       164

confusion matrix:[[81  3]
 [79  1]]
              precision    recall  f1-score   support

          -1       0.48      0.18      0.27        65
          +1       0.51      0.81      0.62        68

    accuracy                           0.50       133
   macro avg       0.49      0.50      0.45       133
weighted avg       0.49      0.50      0.45       133

confusion matrix:[[12 53]
 [13 55]]
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00        72
          +1       0.49      1.00      0.65        68

    accuracy                           0.49       140
   macro avg       0.24      0.50      0.33       140
we

/opt/homebrew/Caskroom/miniconda/base/envs/bdmi/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/Caskroom/miniconda/base/envs/bdmi/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/Caskroom/miniconda/base/envs/bdmi/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

## Step 3: Run with GNB

In [39]:
from sklearn.naive_bayes import GaussianNB

gnb_results = {}

for stock_name, data in processed_data.items():
    X_train = data['X_train_scaled']
    X_test = data['X_test_scaled']
    y_train = data['y_train']
    y_test = data['y_test']

    gnb = GaussianNB()
    gnb.fit(X_train, y_train)
    y_pred = gnb.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(classification_report(y_test, y_pred, target_names=["-1","+1"]))
    cm = confusion_matrix(y_test, y_pred)
    print(f"confusion matrix:{cm}")

              precision    recall  f1-score   support

          -1       0.51      1.00      0.68        84
          +1       0.00      0.00      0.00        80

    accuracy                           0.51       164
   macro avg       0.26      0.50      0.34       164
weighted avg       0.26      0.51      0.35       164

confusion matrix:[[84  0]
 [80  0]]
              precision    recall  f1-score   support

          -1       0.48      0.31      0.37        65
          +1       0.51      0.68      0.58        68

    accuracy                           0.50       133
   macro avg       0.49      0.49      0.48       133
weighted avg       0.49      0.50      0.48       133

confusion matrix:[[20 45]
 [22 46]]
              precision    recall  f1-score   support

          -1       0.53      0.14      0.22        72
          +1       0.49      0.87      0.62        68

    accuracy                           0.49       140
   macro avg       0.51      0.50      0.42       140
we

/opt/homebrew/Caskroom/miniconda/base/envs/bdmi/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/Caskroom/miniconda/base/envs/bdmi/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/Caskroom/miniconda/base/envs/bdmi/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

## Step 4: Run with DecisionTree

In [47]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

for stock_name, data in processed_data.items():
    X_train = data['X_train_scaled']
    X_test = data['X_test_scaled']
    y_train = data['y_train']
    y_test = data['y_test']

    dt = DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42
    )

    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(classification_report(y_test, y_pred, target_names=["-1","+1"]))
    cm = confusion_matrix(y_test, y_pred)
    print(f"confusion matrix:{cm}")

              precision    recall  f1-score   support

          -1       0.51      0.98      0.67        84
          +1       0.33      0.01      0.02        80

    accuracy                           0.51       164
   macro avg       0.42      0.49      0.35       164
weighted avg       0.42      0.51      0.35       164

confusion matrix:[[82  2]
 [79  1]]
              precision    recall  f1-score   support

          -1       0.45      0.58      0.51        65
          +1       0.44      0.31      0.36        68

    accuracy                           0.44       133
   macro avg       0.44      0.45      0.43       133
weighted avg       0.44      0.44      0.43       133

confusion matrix:[[38 27]
 [47 21]]
              precision    recall  f1-score   support

          -1       0.49      0.43      0.46        72
          +1       0.47      0.53      0.50        68

    accuracy                           0.48       140
   macro avg       0.48      0.48      0.48       140
we